# Comparação lexical, semântica e diff bruto

Demonstração sintética: hash, shingles, MinHash, âncoras raras, regra cosmética, extração simples e decisão. Os cortes são ilustrativos. O demo semântico usa um pequeno mapa de conceitos escrito à mão; não finge ser o modelo multilingual-e5-base.

In [1]:
import re
import math
import hashlib
from difflib import SequenceMatcher

# Coleção de referência histórica autorizada, separada dos documentos de teste.
# É um corpus de brinquedo (4 documentos) só para demonstrar frequência/IDF; em produção seria uma coleção grande.
reference_corpus=[
    'Caso locatício antigo: Ante o exposto, requer o regular prosseguimento do feito, com observância das formalidades legais. Por tais razões, requer acolhimento integral dos pedidos formulados na petição inicial. A controvérsia tratava da entrega de imóvel comercial.',
    'Caso previdenciário antigo: Ante o exposto, requer o regular prosseguimento do feito, com observância das formalidades legais. Por tais razões, requer acolhimento integral dos pedidos formulados na petição inicial. O pedido tratava de benefício por incapacidade.',
    'Caso trabalhista antigo: Ante o exposto, requer o regular prosseguimento do feito, com observância das formalidades legais. Por tais razões, requer acolhimento integral dos pedidos formulados na petição inicial. A discussão envolvia horas extras e verbas rescisórias.',
    'Caso de consumo antigo: Ante o exposto, requer o regular prosseguimento do feito, com observância das formalidades legais. Por tais razões, requer acolhimento integral dos pedidos formulados na petição inicial. A parte contestava cobrança de serviço de telefonia.'
]

def tokens(text):
    # Pontuação é token próprio para não colar sinais às palavras.
    return re.findall(r"r\$\s*\d{1,3}(?:\s+\d{1,3})*(?:\.\d{3})*,\d{2}|r\$\s*[\d.]+,\d{2}|\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

def shingles(text, k=5):
    # Não cria shingles atravessando a fronteira entre parágrafos.
    result=set()
    for paragraph in text.splitlines():
        ts=tokens(paragraph)
        result.update(tuple(ts[i:i+k]) for i in range(max(0,len(ts)-k+1)))
    return result

def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0

def containment(a, b):
    return len(a & b) / len(a) if a else 0.0

def minhash_signature(shingle_set, n=128):
    # n funções hash semeadas; a estimativa é a fração de posições iguais.
    sig = []
    for seed in range(n):
        vals = [int.from_bytes(hashlib.blake2b((str(seed)+repr(s)).encode(), digest_size=8).digest(), 'big') for s in shingle_set]
        sig.append(min(vals) if vals else 2**64-1)
    return sig

def normalize_cosmetic(text):
    # Subconjunto demonstrado: caixa, espaços, aspas e travessões; pontuação/paginação requerem diff de produção.
    text = text.casefold().translate(str.maketrans({'“':'"','”':'"','„':'"','‘':"'",'’':"'",'–':'-','—':'-','−':'-'}))
    return re.sub(r"\s+", " ", text).strip()

def is_cosmetic(a, b):
    return a != b and normalize_cosmetic(a) == normalize_cosmetic(b)

def raw_diff(a, b):
    ta, tb = tokens(a), tokens(b)
    parts=[]
    for tag, i1, i2, j1, j2 in SequenceMatcher(a=ta, b=tb, autojunk=False).get_opcodes():
        if tag == 'equal': parts.append(' '.join(ta[i1:i2]))
        elif tag in ('delete','replace'): parts.append('[- ' + ' '.join(ta[i1:i2]) + ' -]')
        if tag in ('insert','replace'): parts.append('[+ ' + ' '.join(tb[j1:j2]) + ' +]')
    return ' '.join(parts)

def inserted_negations(a, b):
    # Procura termos de negação apenas na porção inserida/substituta do texto B.
    ta, tb = tokens(a), tokens(b)
    found=[]
    negations={'não','exceto','salvo'}
    for tag, i1, i2, j1, j2 in SequenceMatcher(a=ta, b=tb, autojunk=False).get_opcodes():
        if tag in ('insert','replace'):
            found.extend(t for t in tb[j1:j2] if t in negations)
    return found

def currency_value(text):
    # Simplificação didática: retorna somente o primeiro valor monetário encontrado.
    m = re.search(r'R\$\s*([\d.]+,\d{2})', text)
    return m.group(1) if m else None

def alignment_coverage(a, b):
    # Conta só trechos alinhados com pelo menos 10 tokens; palavras funcionais isoladas não inflacionam cobertura.
    ta,tb=tokens(a),tokens(b)
    matched=sum(m.size for m in SequenceMatcher(a=ta,b=tb,autojunk=False).get_matching_blocks() if m.size>=10)
    return matched/max(len(ta),len(tb),1)

def rare_blocks_aligned(a, b, reference_docs, min_tokens=30, min_blocks=3):
    # Critério didático: 3+ blocos ordenados (>=30 tokens); em cada bloco >=50% dos shingles têm IDF >=1.5.
    ref_sets=[shingles(doc) for doc in reference_docs]
    n_docs=max(1,len(ref_sets))
    freq={}
    for ss in ref_sets:
        for s in ss: freq[s]=freq.get(s,0)+1
    ta, tb=tokens(a),tokens(b)
    matches=SequenceMatcher(a=ta,b=tb,autojunk=False).get_matching_blocks()
    qualifying=[]
    for m in matches:
        if m.size < min_tokens: continue
        block_shingles={tuple(ta[k:k+5]) for k in range(m.a,m.a+m.size-4)}
        high_idf=sum(math.log((n_docs+1)/(freq.get(s,0)+1))+1 >= 1.5 for s in block_shingles)
        if block_shingles and high_idf/len(block_shingles) >= 0.50:
            qualifying.append((m.a,m.b,m.size))
    # SequenceMatcher emits matching blocks in document order; require >=3 anchors.
    return len(qualifying)>=min_blocks, qualifying

def strip_boilerplate(text, refs, k=5, min_doc_fraction=0.5):
    # Remove tokens covered by k-shingles frequent across the authorized reference collection.
    ref_sets=[shingles(doc,k) for doc in refs]; freq={}
    for ss in ref_sets:
        for sh in ss: freq[sh]=freq.get(sh,0)+1
    common={sh for sh,count in freq.items() if ref_sets and count/len(ref_sets)>=min_doc_fraction}
    ts=tokens(text); remove=set()
    for i in range(max(0,len(ts)-k+1)):
        if tuple(ts[i:i+k]) in common: remove.update(range(i,i+k))
    return ' '.join(tok for i,tok in enumerate(ts) if i not in remove)

def derivation_signal(a, b, reference_docs):
    A, B = shingles(a), shingles(b)
    score_rule = jaccard(A,B) >= 0.95 and containment(A,B) >= 0.98 and containment(B,A) >= 0.98
    block_rule, blocks = rare_blocks_aligned(a,b,reference_docs)
    # Versão abreviada: o Jaccard cai, mas os blocos raros alinhados sustentam a derivação.
    return score_rule or block_rule, score_rule, blocks

def classify(j, alignment_coverage, derivation_supported, has_text_diff,
             cosmetic_only, shingle_count_a, shingle_count_b, min_shingles=10,
             extraction_ok=True, hash_equal=False):
    if hash_equal:
        return 'cópia'  # Hash idêntico dos bytes do arquivo.
    if not extraction_ok or min(shingle_count_a,shingle_count_b) < min_shingles:
        return 'revisão necessária'
    if derivation_supported:
        if not has_text_diff:
            return 'cópia'
        if cosmetic_only:
            return 'cópia com ressalva'
        if alignment_coverage >= 0.60:
            return 'versão'
        return 'revisão necessária'  # Reescrita extensa: limiar inicial a calibrar.
    if j < 0.20 and alignment_coverage < 0.20:
        return 'diferente'  # Exemplo inicial a calibrar.
    return 'revisão necessária'

print('Funções prontas: diff, cosméticos, negações e sinais de derivação são calculados dos textos.')

Funções prontas: diff, cosméticos, negações e sinais de derivação são calculados dos textos.


## 1. Duas petições com o mesmo tema, nome e valor alterados

O exemplo é escrito em quatro parágrafos distintos, não repetido em lote. A tokenização mantém valores monetários inteiros; as três âncoras raras alinham o par apesar de Jaccard baixo.

In [2]:
peticao_a = """A autora contratou a empresa ré para executar serviços de manutenção predial em unidades comerciais durante o primeiro trimestre deste ano. Maria Silva informa que o valor cobrado é R$ 12.500,00 e pede pagamento com juros legais.
O contrato estabeleceu cronograma de atendimento, padrões de qualidade e emissão de notas fiscais após aceite formal pelo responsável técnico. O serviço ocorreu no edifício Central, com troca de bombas hidráulicas e revisão elétrica.
A autora afirma que concluiu as etapas previstas e encaminhou relatórios de execução, recibos de materiais e registros fotográficos ao departamento financeiro. A nota fiscal foi enviada em abril, e a empresa confirmou o recebimento por e-mail.
Apesar das cobranças administrativas e das tentativas de conciliação, não houve quitação integral da obrigação nem apresentação de justificativa documental. Requer condenação pelo saldo em aberto, correção monetária e honorários previstos no contrato. Alega ainda que o atraso comprometeu a compra de materiais para outro serviço já contratado e juntou recibos de peças substituídas, por planilha assinada."""
peticao_b = """A autora contratou a empresa ré para executar serviços de manutenção predial em unidades comerciais durante o primeiro trimestre deste ano. Marina Souza informa que o valor cobrado é R$ 18.900,00 e pede pagamento com juros legais.
O contrato estabeleceu cronograma de atendimento, padrões de qualidade e emissão de notas fiscais após aceite formal pelo responsável técnico. O serviço ocorreu no edifício Central, com substituição de elevadores e pintura das áreas comuns.
A autora afirma que concluiu as etapas previstas e encaminhou relatórios de execução, recibos de materiais e registros fotográficos ao departamento financeiro. A nota fiscal foi enviada em abril, e a empresa acusou recebimento no sistema interno.
Apesar das cobranças administrativas e das tentativas de conciliação, não houve quitação integral da obrigação nem apresentação de justificativa documental. Requer condenação pelo saldo em aberto, correção monetária e honorários previstos no contrato. Sustenta que o atraso impediu a compra de peças para atividade de manutenção em outro edifício e comprovou custos dos reparos executados, por registro digital."""
A, B = shingles(peticao_a), shingles(peticao_b)
j_curto = jaccard(A,B)
sig_a, sig_b = minhash_signature(A), minhash_signature(B)
minhash_curto = sum(x == y for x,y in zip(sig_a,sig_b)) / len(sig_a)
refs=reference_corpus
derivacao_curta, regra_jaccard_curta, blocos_curtos = derivation_signal(peticao_a,peticao_b,refs)
campos_a = {'parte':'Maria Silva', 'valor':currency_value(peticao_a)}
campos_b = {'parte':'Marina Souza', 'valor':currency_value(peticao_b)}
campos_alterados = {k for k in campos_a if campos_a[k] != campos_b[k]}
diff_curto = raw_diff(peticao_a,peticao_b)
diff_textual_curto = peticao_a != peticao_b
cosmetico_curto = is_cosmetic(peticao_a,peticao_b)
coverage_curto = alignment_coverage(peticao_a,peticao_b)
print(f'Jaccard exato: {j_curto:.3f}; MinHash (128 funções): {minhash_curto:.3f}')
print('Alterações no diff bruto:', ' '.join(re.findall(r'\[- .*? -\]|\[\+ .*? \+\]',diff_curto)))
print('Campos para destaque:', sorted(campos_alterados))
print(f'Derivação calculada: {derivacao_curta}; Jaccard rule: {regra_jaccard_curta}; rare aligned blocks: {len(blocos_curtos)}; coverage: {coverage_curto:.3f}')
classe_curta = classify(j_curto,coverage_curto,derivacao_curta,diff_textual_curto,cosmetico_curto,len(A),len(B))
print(f'classe = "{classe_curta}"')

Jaccard exato: 0.516; MinHash (128 funções): 0.469
Alterações no diff bruto: [- maria silva -] [+ marina souza +] [- r$ 12.500,00 -] [+ r$ 18.900,00 +] [- troca -] [+ substituição +] [- bombas hidráulicas -] [+ elevadores +] [- revisão elétrica -] [+ pintura das áreas comuns +] [- confirmou o -] [+ acusou +] [- por e - mail -] [+ no sistema interno +] [- alega ainda -] [+ sustenta +] [- comprometeu -] [+ impediu +] [- materiais -] [+ peças +] [+ atividade de manutenção em +] [- serviço já contratado -] [+ edifício +] [- juntou recibos de peças substituídas -] [+ comprovou custos dos reparos executados +] [- planilha assinada -] [+ registro digital +]
Campos para destaque: ['parte', 'valor']
Derivação calculada: True; Jaccard rule: False; rare aligned blocks: 3; coverage: 0.731
classe = "versão"


## 2. Documento mais longo: ramo dos três blocos raros

O parágrafo adicional varia entre as seções. O ramo lexical falha para este par; quatro blocos específicos alinhados sustentam derivação.

In [3]:
# Parágrafos adicionais diferentes entre si; este par demonstra o ramo de três blocos, não só o corte de Jaccard.
complementos = [
    'Os documentos incluem ordens de serviço assinadas, fotografias datadas das intervenções e comprovantes de compra dos componentes utilizados nas unidades atendidas.',
    'A empresa ré recebeu os relatórios em cada etapa e solicitou pequenos ajustes, concluídos antes da vistoria final feita pelo encarregado predial.',
    'As partes trocaram mensagens sobre o cronograma e a liberação de acesso aos locais de trabalho, sem reclamação sobre o resultado das intervenções.'
]
doc_longo_a='\n'.join([peticao_a]+complementos)
doc_longo_b='\n'.join([peticao_b]+complementos)
LA,LB=shingles(doc_longo_a),shingles(doc_longo_b)
j_longo=jaccard(LA,LB)
derivacao_longa,regra_jaccard_longa,blocos_longos=derivation_signal(doc_longo_a,doc_longo_b,refs)
cobertura_longa=alignment_coverage(doc_longo_a,doc_longo_b)
print(f'Jaccard: {j_longo:.3f}; ramo lexical: {regra_jaccard_longa}')
print('Blocos raros alinhados:',[(n_tokens) for _,_,n_tokens in blocos_longos])
print('Derivação pelo ramo de blocos:',derivacao_longa and not regra_jaccard_longa)
classe_longa=classify(j_longo,cobertura_longa,derivacao_longa,doc_longo_a!=doc_longo_b,is_cosmetic(doc_longo_a,doc_longo_b),len(LA),len(LB))
print(f'classe = "{classe_longa}"')

complementos_extensos=[
    'O responsável pelo imóvel acompanhou as intervenções em datas diferentes, registrou a entrada de equipes e confirmou a substituição dos materiais previstos na ordem de serviço.',
    'A autora juntou notas fiscais de fornecedores, comprovantes bancários de aquisição e fotografias que mostram o estado das instalações antes e depois dos reparos.',
    'A ré pediu prazo adicional para conferência dos documentos e recebeu cópia dos relatórios, sem apontar divergência de quantidade, preço ou qualidade no período de análise.',
    'As tratativas foram retomadas por mensagem eletrônica e reunião remota, mas não houve acordo sobre o calendário para pagamento das faturas vencidas.',
    'A petição delimita o pedido ao saldo documentalmente comprovado, com atualização desde o vencimento e encargos previstos no instrumento contratual.',
    'O síndico acompanhou as visitas técnicas, liberou acesso às salas de máquinas e assinou os registros de conclusão encaminhados pela prestadora ao setor administrativo.',
    'A planilha anexada relaciona cada atividade ao número da ordem correspondente e discrimina materiais, horas de trabalho e datas de aceite pelo cliente.',
    'A empresa recebeu aviso de cobrança com prazo para resposta e solicitou cópia adicional das notas, que foi enviada pelo departamento financeiro no mesmo dia.',
    'Em audiência de conciliação, as partes concordaram sobre a execução dos reparos, mas divergiram quanto ao saldo e à incidência dos encargos contratuais.'
]
doc_extenso_a='\n'.join([doc_longo_a]+complementos_extensos)


Jaccard: 0.620; ramo lexical: False
Blocos raros alinhados: [37, 36, 38, 73]
Derivação pelo ramo de blocos: True
classe = "versão"


## 3. Reescrita sem mudança de campo crítico

O diff e a cobertura são calculados dos textos; não é necessário detectar uma alteração de campo para que a mudança textual resulte em versão.

In [4]:
doc_reescrita=doc_extenso_a.replace('encaminhou relatórios de execução','enviou relatórios da execução',1)
R1,R2=shingles(doc_extenso_a),shingles(doc_reescrita)
j_reescrita=jaccard(R1,R2)
derivacao_reescrita,_,blocos_reescrita=derivation_signal(doc_extenso_a,doc_reescrita,refs)
coverage_reescrita=alignment_coverage(doc_extenso_a,doc_reescrita)
print(f'Jaccard: {j_reescrita:.3f}; campos críticos extraídos alterados: nenhum')
print('Diff não cosmético calculado:',doc_extenso_a!=doc_reescrita and not is_cosmetic(doc_extenso_a,doc_reescrita))
print('Derivação e cobertura:',derivacao_reescrita,f'{coverage_reescrita:.3f}')
classe_reescrita=classify(j_reescrita,coverage_reescrita,derivacao_reescrita,doc_extenso_a!=doc_reescrita,is_cosmetic(doc_extenso_a,doc_reescrita),len(R1),len(R2))
print(f'classe = "{classe_reescrita}"')

Jaccard: 0.967; campos críticos extraídos alterados: nenhum
Diff não cosmético calculado: True
Derivação e cobertura: True 0.994
classe = "versão"


## 4. Negação inserida sem alterar o valor

SequenceMatcher localiza “não” na região inserida; o valor monetário permanece igual. O diff bruto bloqueia cópia.

In [5]:
doc_negacao_a=doc_extenso_a.replace('não houve quitação integral','houve quitação integral',1)
doc_negacao_b=doc_extenso_a
NA,NB=shingles(doc_negacao_a),shingles(doc_negacao_b)
j_negacao=jaccard(NA,NB)
negacoes_inseridas=inserted_negations(doc_negacao_a,doc_negacao_b)
derivacao_negacao,_,_=derivation_signal(doc_negacao_a,doc_negacao_b,refs)
coverage_negacao=alignment_coverage(doc_negacao_a,doc_negacao_b)
print(f'Jaccard: {j_negacao:.3f}; negação inserida detectada: {negacoes_inseridas}')
print('Valor extraído mudou:',currency_value(doc_negacao_a)!=currency_value(doc_negacao_b))
classe_negacao=classify(j_negacao,coverage_negacao,derivacao_negacao,doc_negacao_a!=doc_negacao_b,is_cosmetic(doc_negacao_a,doc_negacao_b),len(NA),len(NB))
print(f'classe = "{classe_negacao}"')

Jaccard: 0.979; negação inserida detectada: ['não']
Valor extraído mudou: False
classe = "versão"


## 5. Cópia cosmética, espaço numérico e texto insuficiente

A normalização cosmética demonstrada só aceita caixa, espaços colapsáveis, aspas e travessões tipográficos. Espaço dentro de número e junção de palavras são diferenças materiais; abaixo de 10 shingles, a saída é revisão.

In [6]:
# Testes de borda: as entradas de classify() abaixo (vazio/curto) são escolhidas à mão para exercitar a regra.
texto_base='\n'.join([peticao_a]+complementos)
texto_cosmetico=texto_base.upper().replace('\n','\n  ')
derivacao_cosmetica,_,_=derivation_signal(texto_base,texto_cosmetico,refs)
coverage_cosmetica=alignment_coverage(texto_base,texto_cosmetico)
print('Caixa/espaços diferem; is_cosmetic:',is_cosmetic(texto_base,texto_cosmetico))
print('Resultado cosmético:',classify(jaccard(shingles(texto_base),shingles(texto_cosmetico)),coverage_cosmetica,derivacao_cosmetica,True,is_cosmetic(texto_base,texto_cosmetico),len(shingles(texto_base)),len(shingles(texto_cosmetico))))
base_num_a=doc_extenso_a
base_num_b=doc_extenso_a.replace('R$ 12.500,00','R$ 1 2.500,00',1)
NA,NB=shingles(base_num_a),shingles(base_num_b)
derivacao_num,_,_=derivation_signal(base_num_a,base_num_b,refs)
print('Valor com espaço interno: is_cosmetic =',is_cosmetic(base_num_a,base_num_b),'; classe =',classify(jaccard(NA,NB),alignment_coverage(base_num_a,base_num_b),derivacao_num,base_num_a!=base_num_b,is_cosmetic(base_num_a,base_num_b),len(NA),len(NB)))
print('a ré pagou -> aré pagou: is_cosmetic =',is_cosmetic('a ré pagou','aré pagou'))
print('Texto vazio (0 shingles):',classify(0,0,False,False,False,0,0))
print('Texto curto (<10 shingles):',classify(0,0,False,False,False,2,2))
sections=doc_extenso_a.splitlines()
doc_abreviado='\n'.join(s for i,s in enumerate(sections) if i not in (1,4))
SA,SB=shingles(doc_extenso_a),shingles(doc_abreviado)
derivacao_abrev,_,blocos_abrev=derivation_signal(doc_extenso_a,doc_abreviado,refs)
contencao_curta=containment(SB,SA)
cobertura_abrev=alignment_coverage(doc_extenso_a,doc_abreviado)
print(f'Abreviada: contenção na direção curta→longa={contencao_curta:.3f}; blocos={len(blocos_abrev)}; cobertura={cobertura_abrev:.3f}')
print('Classe da abreviada:',classify(jaccard(SA,SB),cobertura_abrev,derivacao_abrev,doc_extenso_a!=doc_abreviado,is_cosmetic(doc_extenso_a,doc_abreviado),len(SA),len(SB)))

Caixa/espaços diferem; is_cosmetic: True
Resultado cosmético: cópia com ressalva
Valor com espaço interno: is_cosmetic = False ; classe = versão
a ré pagou -> aré pagou: is_cosmetic = False
Texto vazio (0 shingles): revisão necessária
Texto curto (<10 shingles): revisão necessária
Abreviada: contenção na direção curta→longa=1.000; blocos=3; cobertura=0.875
Classe da abreviada: versão


## 6. Mesmo boilerplate, fatos diferentes

A coleção de referência é independente dos dois pares de teste. Remover shingles frequentes reduz a semelhança e evita confundir texto padrão com derivação.

In [7]:
# Corpus de referência histórica independente dos dois documentos de teste.
refs_boilerplate=reference_corpus
bloco_padrao='Ante o exposto, requer o regular prosseguimento do feito, com observância das formalidades legais. Por tais razões, requer acolhimento integral dos pedidos formulados na petição inicial.'
doc_a=bloco_padrao+' A autora discute contrato de locação residencial firmado em janeiro, entrega das chaves, reparos no imóvel e devolução do depósito caução após vistoria detalhada.'
doc_b=bloco_padrao+' O requerente contesta multa de trânsito aplicada em rodovia estadual, questiona a sinalização do trecho e pede cancelamento dos pontos lançados na carteira.'
A_bruto,B_bruto=shingles(doc_a),shingles(doc_b); j_antes=jaccard(A_bruto,B_bruto)
limpo_a,limpo_b=strip_boilerplate(doc_a,refs_boilerplate),strip_boilerplate(doc_b,refs_boilerplate)
A_limpo,B_limpo=shingles(limpo_a),shingles(limpo_b); j_depois=jaccard(A_limpo,B_limpo)
cov=alignment_coverage(limpo_a,limpo_b)
classe=classify(j_depois,cov,False,limpo_a!=limpo_b,False,len(A_limpo),len(B_limpo))
print('Referências: 4 documentos históricos separados dos pares de teste.')
print(f'Jaccard antes de retirar boilerplate: {j_antes:.3f}')
print(f'Jaccard depois de retirar boilerplate: {j_depois:.3f}; cobertura residual: {cov:.3f}')
print(f'Shingles restantes: {len(A_limpo)}, {len(B_limpo)}')
print(f'classe = "{classe}"')

Referências: 4 documentos históricos separados dos pares de teste.


Jaccard antes de retirar boilerplate: 0.342
Jaccard depois de retirar boilerplate: 0.000; cobertura residual: 0.000
Shingles restantes: 22, 21
classe = "diferente"


## 7. Busca semântica para paráfrases (e seu limite)

Vetores didáticos usam um pequeno mapa de sinônimos para mostrar a ideia; não são o modelo e5. Na implementação, avaliaria um modelo multilíngue em dados rotulados. O segundo exemplo mostra por que o embedding só recupera candidatos: uma negação inserida quase não muda o vetor, e quem detecta a mudança é o diff bruto.

In [8]:
parafrase_a='A autora solicita indenização pelos danos provocados.'
parafrase_b='A requerente pede reparação pelo prejuízo causado.'
axes=['parte','pedido','indenização','dano','causa']
conceptos={'autora':'parte','requerente':'parte','solicita':'pedido','pede':'pedido','indenização':'indenização','reparação':'indenização','danos':'dano','prejuízo':'dano','provocados':'causa','causado':'causa'}
conceptos.update({'ré':'parte','pagou':'pedido','dívida':'dano'})
def toy_embedding(text):
    v=[0.0]*len(axes)
    for token in tokens(text):
        concept=conceptos.get(token)
        if concept: v[axes.index(concept)]+=1
    norm=math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/norm for x in v]
def cosine(a,b): return sum(x*y for x,y in zip(a,b))
J_parafrase=jaccard(shingles(parafrase_a),shingles(parafrase_b))
cos_parafrase=cosine(toy_embedding(parafrase_a),toy_embedding(parafrase_b))
print(f'Jaccard lexical (shingles): {J_parafrase:.3f}')
print(f'Similaridade vetorial didática: {cos_parafrase:.3f}; candidato semântico: {cos_parafrase >= 0.80}')
print('Limite: mapa manual para demonstrar o princípio; não mede a qualidade de multilingual-e5-base.')
# Limite: negação não altera o vetor, mas inverte o sentido.
neg_a='A ré pagou a dívida no prazo combinado.'
neg_b='A ré não pagou a dívida no prazo combinado.'
cos_neg=cosine(toy_embedding(neg_a),toy_embedding(neg_b))
print(f'Negação: similaridade vetorial didática = {cos_neg:.3f}; Jaccard = {jaccard(shingles(neg_a),shingles(neg_b)):.3f}')
print('Negação detectada pelo diff bruto:',inserted_negations(neg_a,neg_b),'; só cosmético:',is_cosmetic(neg_a,neg_b))
print('Embedding sozinho não separa as duas frases; o diff bruto impede “cópia”.')


Jaccard lexical (shingles): 0.000
Similaridade vetorial didática: 1.000; candidato semântico: True
Limite: mapa manual para demonstrar o princípio; não mede a qualidade de multilingual-e5-base.
Negação: similaridade vetorial didática = 1.000; Jaccard = 0.375
Negação detectada pelo diff bruto: ['não'] ; só cosmético: False
Embedding sozinho não separa as duas frases; o diff bruto impede “cópia”.


## 8. Avaliação sintética: sistema e baseline

Os pares cobrem cópia, cópia cosmética, quatro tipos de versão, boilerplate comum e três casos de revisão (reescrita extensa, texto curto, sobreposição parcial). O notebook também informa quantos cenários são realmente distintos e qual ramo de derivação decidiu cada par. A matriz mede estas fixtures, não desempenho em petições reais. O baseline hash + Jaccard não tem as classes “cópia com ressalva” e “revisão necessária”; os erros nelas refletem essa ausência.

In [9]:
# Base longa e variada para mutações isoladas em pares de versão.
base_eval=doc_extenso_a
pairs=[]
def add_pair(name,a,b,label): pairs.append((name,a,b,label))
for i in range(6): add_pair(f'cópia exata {i+1}',base_eval,base_eval,'cópia')
for i in range(4): add_pair(f'cópia cosmética {i+1}',base_eval,base_eval.upper().replace('\n','\n  '),'cópia com ressalva')
# Oito versões: alterações isoladas de valor, negação, frase e abreviação.
value_variants=['R$ 18.900,00','R$ 14.250,00']
for i,v in enumerate(value_variants): add_pair(f'versão valor {i+1}',base_eval,base_eval.replace('R$ 12.500,00',v,1),'versão')
neg_variants=[('não houve quitação integral','houve quitação integral'),('a empresa confirmou o recebimento','a empresa não confirmou o recebimento')]
for i,(old,new) in enumerate(neg_variants): add_pair(f'versão negação {i+1}',base_eval,base_eval.replace(old,new,1),'versão')
rewrite_variants=[('encaminhou relatórios de execução','enviou relatórios da execução'),('confirmou o recebimento','registrou o recebimento')]
for i,(old,new) in enumerate(rewrite_variants): add_pair(f'versão reescrita {i+1}',base_eval,base_eval.replace(old,new,1),'versão')
for i,omit in enumerate([(1,4),(2,6)]):
    shortened='\n'.join(s for j,s in enumerate(base_eval.splitlines()) if j not in omit)
    add_pair(f'versão abreviada {i+1}',base_eval,shortened,'versão')
# Independent fact pairs share frequent template language but not derivation.
boiler='Ante o exposto, requer o regular prosseguimento do feito, com observância das formalidades legais. Por tais razões, requer acolhimento integral dos pedidos formulados na petição inicial.'
facts=[
('A autora discute locação residencial, entrega das chaves e retenção do depósito caução após vistoria.','O condutor contesta autuação em rodovia, sinalização insuficiente e pontos lançados na carteira.'),
('O consumidor questiona cobrança de telefonia, pacote não contratado e restituição de tarifas debitadas.','A segurada pede benefício por incapacidade com base em exames e histórico médico recente.'),
('A trabalhadora cobra horas extras, intervalo não concedido e diferenças no cálculo das verbas rescisórias.','O proprietário discute divisa rural, levantamento topográfico e passagem de acesso ao terreno.'),
('O passageiro pede reparação por bagagem extraviada e despesas durante conexão internacional.','A empresa impugna multa ambiental relativa a licença de operação e relatório de fiscalização.'),
('O paciente questiona negativa de cobertura para procedimento indicado pelo médico assistente.','O comprador pede abatimento por defeito em equipamento entregue fora das especificações.'),
('A autora requer alimentos provisórios e apresenta comprovantes de renda e despesas escolares.','O réu contesta cobrança de condomínio e apresenta recibos de quitação das parcelas.'),
]
for i,(fa,fb) in enumerate(facts):
    extra_a=' A autora juntou comprovantes individualizados, cronologia dos eventos, correspondência eletrônica e recibos que detalham os valores reclamados no processo.'
    extra_b=' A parte anexou relatórios técnicos, registros independentes, datas de fiscalização e documentos administrativos relativos à controvérsia apresentada ao juízo.'
    add_pair(f'diferente com boilerplate {i+1}',boiler+' '+fa+extra_a,boiler+' '+fb+extra_b,'diferente')

# Três pares que devem ir à revisão: reescrita extensa, texto curto e sobreposição parcial.
paras=base_eval.splitlines()
novos=[
 'O contrato previa vistorias mensais, mas as partes nunca registraram em ata o resultado de cada visita ao local.',
 'A empresa alega que parte dos equipamentos já estava danificada antes da intervenção e pede perícia independente.',
 'Segundo a defesa, o cronograma foi alterado por solicitação verbal do síndico, sem aditivo contratual formal.',
 'A cobrança inclui deslocamento de equipes em feriados, item que a ré afirma não ter autorizado previamente.',
 'O laudo particular anexado menciona apenas parte das unidades e não esclarece quem forneceu os materiais.',
 'As testemunhas arroladas trabalharam em períodos distintos e divergem sobre a data de conclusão dos serviços.',
 'A ré sustenta que a multa contratual não incide, porque o atraso decorreu de falta de acesso às áreas técnicas.',
 'Há divergência sobre o índice de correção aplicável, já que o contrato cita dois indicadores diferentes.',
 'O pedido de tutela depende da comprovação do risco de dano, que a autora diz estar demonstrado pelas fotografias.',
 'A discussão sobre honorários envolve percentual fixado em cláusula específica e eventual redução equitativa.',
 'O juízo foi informado de que as partes tentaram mediação extrajudicial, sem sucesso, antes do ajuizamento.',
]
it=iter(novos)
reescrita_extensa='\n'.join(p if i in {0,1,3,4,6} else next(it) for i,p in enumerate(paras))
add_pair('revisão reescrita extensa',base_eval,reescrita_extensa,'revisão necessária')
add_pair('revisão texto curto','O réu pagou o valor.','O réu não pagou o valor.','revisão necessária')
parcial_a='\n'.join(paras[:2]+paras[7:11])
parcial_b='\n'.join(paras[:2]+novos[4:8])
add_pair('revisão sobreposição parcial',parcial_a,parcial_b,'revisão necessária')

def system_predict(a,b):
    fa,fb=strip_boilerplate(a,refs_boilerplate),strip_boilerplate(b,refs_boilerplate)
    A,B=shingles(fa),shingles(fb); j=jaccard(A,B); cov=alignment_coverage(fa,fb)
    deriv,score_rule,blocks=derivation_signal(fa,fb,refs_boilerplate)
    ramo='Jaccard' if score_rule else ('blocos' if deriv else '-')
    # hash_equal usa igualdade de texto; em produção seria o hash dos bytes do arquivo.
    return classify(j,cov,deriv,a!=b,is_cosmetic(a,b),len(A),len(B),hash_equal=(a==b)),j,cov,ramo

def baseline_predict(a,b):
    if a==b: return 'cópia'
    j=jaccard(shingles(a),shingles(b))
    if j>=0.95: return 'cópia'
    if j>=0.20: return 'versão'
    return 'diferente'

labels=['cópia','cópia com ressalva','versão','diferente','revisão necessária']
records=[]
for name,a,b,expected in pairs:
    pred,j,cov,ramo=system_predict(a,b)
    base=baseline_predict(a,b)
    records.append({'scenario':name,'expected':expected,'system':pred,'baseline':base,'jaccard':j,'diff':a!=b,'ramo':ramo})

def print_matrix(which):
    preds=labels
    print('Atual→Predito | '+' | '.join(preds))
    for actual in labels:
        vals=[sum(r['expected']==actual and r[which]==p for r in records) for p in preds]
        print(actual+' | '+' | '.join(map(str,vals)))
print('Pares:',len(records),'; cenários distintos:',len({(a,b) for _,a,b,_ in pairs}))
print('Matriz do sistema:'); print_matrix('system')
print('Matriz baseline hash + Jaccard:'); print_matrix('baseline')
print('Versão→cópia, sistema:',sum(r['expected']=='versão' and r['system']=='cópia' for r in records))
print('Versão→cópia, baseline:',sum(r['expected']=='versão' and r['baseline']=='cópia' for r in records))
fixes=[r['scenario'] for r in records if r['system']==r['expected'] and r['baseline']!=r['expected']]
print('Erros do baseline corrigidos pelo sistema:',', '.join(fixes))
print('Erros do sistema:',[r['scenario'] for r in records if r['system']!=r['expected']])
print('Erros do baseline (total):',sum(r['baseline']!=r['expected'] for r in records))
print('Ramos de derivação usados:',sorted({r['ramo'] for r in records}))
print('\n| Cenário | Jaccard após sinais | Diff | Ramo | Classe |')
print('|---|---:|---|---|---|')
for r in records: print(f"| {r['scenario']} | {r['jaccard']:.3f} | {r['diff']} | {r['ramo']} | {r['system']} |")

Pares: 27 ; cenários distintos: 19
Matriz do sistema:
Atual→Predito | cópia | cópia com ressalva | versão | diferente | revisão necessária
cópia | 6 | 0 | 0 | 0 | 0
cópia com ressalva | 0 | 4 | 0 | 0 | 0
versão | 0 | 0 | 8 | 0 | 0
diferente | 0 | 0 | 0 | 6 | 0
revisão necessária | 0 | 0 | 0 | 0 | 3
Matriz baseline hash + Jaccard:
Atual→Predito | cópia | cópia com ressalva | versão | diferente | revisão necessária
cópia | 6 | 0 | 0 | 0 | 0
cópia com ressalva | 4 | 0 | 0 | 0 | 0
versão | 6 | 0 | 2 | 0 | 0
diferente | 0 | 0 | 6 | 0 | 0
revisão necessária | 0 | 0 | 2 | 1 | 0
Versão→cópia, sistema: 0
Versão→cópia, baseline: 6
Erros do baseline corrigidos pelo sistema: cópia cosmética 1, cópia cosmética 2, cópia cosmética 3, cópia cosmética 4, versão valor 1, versão valor 2, versão negação 1, versão negação 2, versão reescrita 1, versão reescrita 2, diferente com boilerplate 1, diferente com boilerplate 2, diferente com boilerplate 3, diferente com boilerplate 4, diferente com boilerplate 5,